In [ ]:
"""
=============================================================
  OPTIMIZED NSE PORTFOLIO BACKTEST
  - Parallel yfinance fetching (10x faster)
  - Cached data (skip re-downloads on re-run)
  - Bug fixes from original script
  - Full metrics: CAGR, Sharpe, Max Drawdown, Nifty benchmark
=============================================================
"""

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os
import pickle
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

# ─────────────────────────────────────────────
#  SETTINGS  (edit these)
# ─────────────────────────────────────────────
INITIAL_CAPITAL   = 100_000
STOP_LOSS_PCT     = 0.10        # 10% hard stop
MAX_POSITIONS     = 20
LOOKBACK_PERIOD   = "10y"
ATR_PERIOD        = 14          # FIX: original used 1 (too noisy)
ATR_MULTIPLIER    = 2.0         # FIX: original used 1 (too tight)
MAX_DRAWDOWN_PCT  = 0.15        # FIX: original was 0.02 (exits every trade)
MIN_MARKET_CAP_CR = 1_000       # crore INR
MAX_MARKET_CAP_CR = 1_000_000

MAX_WORKERS       = 20          # parallel threads for yfinance
CACHE_DIR         = "cache"     # folder to store downloaded data
USE_CACHE         = True        # set False to force re-download

# ─────────────────────────────────────────────
#  CACHE HELPERS
# ─────────────────────────────────────────────
os.makedirs(CACHE_DIR, exist_ok=True)

def cache_path(ticker):
    return os.path.join(CACHE_DIR, ticker.replace(".", "_") + ".pkl")

def load_from_cache(ticker):
    p = cache_path(ticker)
    if USE_CACHE and os.path.exists(p):
        with open(p, "rb") as f:
            return pickle.load(f)
    return None

def save_to_cache(ticker, data):
    with open(cache_path(ticker), "wb") as f:
        pickle.dump(data, f)

# ─────────────────────────────────────────────
#  LOAD NSE TICKERS
# ─────────────────────────────────────────────
def load_nse_tickers():
    df = pd.read_csv("data/EQUITY_L.csv")
    symbols = df["SYMBOL"].dropna().astype(str).str.strip().unique().tolist()
    return [s + ".NS" for s in symbols if "&" not in s]

# ─────────────────────────────────────────────
#  UT BOT  (fixed ATR period)
# ─────────────────────────────────────────────
def compute_utbot(df, atr_period=ATR_PERIOD, multiplier=ATR_MULTIPLIER):
    df = df.copy()
    df["tr"] = np.maximum(
        df["High"] - df["Low"],
        np.maximum(
            (df["High"] - df["Close"].shift()).abs(),
            (df["Low"]  - df["Close"].shift()).abs()
        )
    )
    df["atr"] = df["tr"].ewm(span=atr_period, adjust=False).mean()  # EMA ATR (smoother)

    df["upper"] = df["Close"] - multiplier * df["atr"]
    df["lower"] = df["Close"] + multiplier * df["atr"]

    trend = [1]
    for i in range(1, len(df)):
        if   df["Close"].iloc[i] > df["lower"].iloc[i - 1]:  trend.append(1)
        elif df["Close"].iloc[i] < df["upper"].iloc[i - 1]:  trend.append(-1)
        else:                                                  trend.append(trend[-1])

    df["trend"]      = trend
    df["buy"]        = (df["trend"] == 1)  & (df["trend"].shift() == -1)
    df["sell"]       = (df["trend"] == -1) & (df["trend"].shift() == 1)
    df["volatility"] = df["Close"].pct_change(fill_method=None).rolling(20).std()
    return df

# ─────────────────────────────────────────────
#  FETCH ONE TICKER  (used inside thread pool)
# ─────────────────────────────────────────────
def fetch_ticker(ticker):
    cached = load_from_cache(ticker)
    if cached is not None:
        return ticker, cached

    try:
        stock = yf.Ticker(ticker)
        info  = stock.info

        market_cap = info.get("marketCap")
        if market_cap is None:
            return ticker, None

        mc_cr = market_cap / 1e7
        if not (MIN_MARKET_CAP_CR <= mc_cr <= MAX_MARKET_CAP_CR):
            return ticker, None

        df = stock.history(period=LOOKBACK_PERIOD)
        if df.empty or len(df) < 60:
            return ticker, None

        df = compute_utbot(df)

        eps    = info.get("trailingEps")
        if eps is None or eps <= 0:
            return ticker, None

        growth = info.get("earningsQuarterlyGrowth") or 0.05
        g      = min(growth * 100, 12)
        intrinsic = eps * (8.5 + 2 * g)

        df["Intrinsic"]     = intrinsic
        df["MarketCapCr"]   = mc_cr

        save_to_cache(ticker, df)
        return ticker, df

    except Exception:
        return ticker, None

# ─────────────────────────────────────────────
#  PARALLEL DATA LOAD
# ─────────────────────────────────────────────
def load_all_data(tickers):
    all_data = {}
    total    = len(tickers)
    done     = 0

    print(f"\nFetching {total} tickers with {MAX_WORKERS} parallel workers …\n")
    t0 = time.time()

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(fetch_ticker, t): t for t in tickers}
        for fut in as_completed(futures):
            ticker, df = fut.result()
            done += 1
            if df is not None:
                all_data[ticker] = df
            if done % 100 == 0 or done == total:
                elapsed = time.time() - t0
                pct     = done / total * 100
                eta     = (elapsed / done) * (total - done)
                print(f"  [{done}/{total}]  {pct:.0f}%  |  loaded: {len(all_data)}  |  ETA: {eta/60:.1f} min")

    print(f"\nLoaded {len(all_data)} stocks in {(time.time()-t0)/60:.1f} min\n")
    return all_data

# ─────────────────────────────────────────────
#  METRICS HELPERS
# ─────────────────────────────────────────────
def calc_cagr(start, end, years):
    if years <= 0 or start <= 0:
        return 0.0
    return (end / start) ** (1 / years) - 1

def calc_max_drawdown(equity):
    eq   = np.array(equity)
    peak = np.maximum.accumulate(eq)
    dd   = (eq - peak) / peak
    return dd.min()

def calc_sharpe(equity, risk_free=0.06):
    eq      = np.array(equity)
    rets    = np.diff(eq) / eq[:-1]
    daily_rf = risk_free / 252
    excess  = rets - daily_rf
    if excess.std() == 0:
        return 0.0
    return (excess.mean() / excess.std()) * np.sqrt(252)

# ─────────────────────────────────────────────
#  BACKTEST
# ─────────────────────────────────────────────
def run_backtest(all_data):
    cash           = float(INITIAL_CAPITAL)
    open_positions = {}
    trade_log      = []
    equity_curve   = []
    date_index     = []

    all_dates = sorted(set(d for df in all_data.values() for d in df.index))

    print(f"Running backtest over {len(all_dates)} trading days …\n")

    for current_date in all_dates:

        # ── SELL ──────────────────────────────
        for ticker in list(open_positions.keys()):
            df  = all_data[ticker]
            if current_date not in df.index:
                continue

            row = df.loc[current_date]
            pos = open_positions[ticker]

            pos["highest_price"] = max(pos["highest_price"], row["Close"])

            stop_price = pos["entry_price"] * (1 - STOP_LOSS_PCT)
            atr_stop   = pos["highest_price"] - (row["atr"] * ATR_MULTIPLIER)
            drawdown   = (pos["highest_price"] - row["Close"]) / pos["highest_price"]

            exit_reason = None
            if   row["Close"] <= stop_price:          exit_reason = "Stop Loss"
            elif row["Close"] <= atr_stop:            exit_reason = "ATR Trailing Stop"
            elif drawdown     >= MAX_DRAWDOWN_PCT:    exit_reason = "Max Drawdown"
            elif row["sell"]:                         exit_reason = "UT Sell Signal"

            if exit_reason is None:
                continue

            exit_price = row["Close"]
            proceeds   = pos["shares"] * exit_price
            profit     = proceeds - pos["invested"]
            cash      += proceeds

            trade_log.append({
                "Stock":        ticker,
                "Entry Date":   pos["entry_date"],
                "Exit Date":    current_date,
                "Entry Price":  round(pos["entry_price"], 2),
                "Exit Price":   round(exit_price, 2),
                "Shares":       round(pos["shares"], 4),
                "Invested":     round(pos["invested"], 2),
                "Profit":       round(profit, 2),
                "Return %":     round((exit_price / pos["entry_price"] - 1) * 100, 2),
                "Exit Reason":  exit_reason,
                "Market Cap Cr": row["MarketCapCr"],
                "Holding Days": (current_date - pos["entry_date"]).days,
            })
            del open_positions[ticker]

        # ── BUY ───────────────────────────────
        available_slots = MAX_POSITIONS - len(open_positions)

        if available_slots > 0 and cash > 0:
            candidates = []

            for ticker, df in all_data.items():
                if ticker in open_positions:
                    continue
                if current_date not in df.index:
                    continue

                row = df.loc[current_date]

                if pd.isna(row["volatility"]) or row["volatility"] == 0:
                    continue
                if not row["buy"]:
                    continue
                if row["Close"] >= row["Intrinsic"]:
                    continue

                discount = (row["Intrinsic"] - row["Close"]) / row["Intrinsic"]
                score    = discount / row["volatility"]
                candidates.append((ticker, score))

            candidates.sort(key=lambda x: x[1], reverse=True)

            for ticker, _ in candidates[:available_slots]:
                # FIX: safe allocation — never divide by zero
                remaining_slots = MAX_POSITIONS - len(open_positions)
                if remaining_slots <= 0 or cash <= 0:
                    break

                allocation = cash / remaining_slots
                row        = all_data[ticker].loc[current_date]
                shares     = allocation / row["Close"]

                cash -= allocation
                open_positions[ticker] = {
                    "entry_date":    current_date,
                    "entry_price":   row["Close"],
                    "highest_price": row["Close"],
                    "shares":        shares,
                    "invested":      allocation,
                }

        # ── DAILY EQUITY ──────────────────────
        portfolio_value = cash
        for ticker, pos in open_positions.items():
            df = all_data[ticker]
            if current_date in df.index:
                portfolio_value += pos["shares"] * df.loc[current_date]["Close"]

        equity_curve.append(portfolio_value)
        date_index.append(current_date)

    return pd.DataFrame(trade_log), equity_curve, date_index

# ─────────────────────────────────────────────
#  BENCHMARK (Nifty 50)
# ─────────────────────────────────────────────
def fetch_nifty(start_date, end_date):
    try:
        nifty = yf.download("^NSEI", start=start_date, end=end_date, progress=False)
        close = nifty["Close"].squeeze()
        norm  = close / close.iloc[0] * INITIAL_CAPITAL
        return norm
    except:
        return None

# ─────────────────────────────────────────────
#  PLOT
# ─────────────────────────────────────────────
def plot_results(equity_curve, date_index, trades, nifty=None):
    fig = plt.figure(figsize=(16, 12))
    gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

    # 1. Equity curve
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(date_index, equity_curve, color="#534AB7", lw=1.5, label="Strategy")
    if nifty is not None:
        nifty_aligned = nifty.reindex(pd.DatetimeIndex(date_index), method="ffill")
        ax1.plot(date_index, nifty_aligned.values, color="#1D9E75", lw=1.2,
                 linestyle="--", label="Nifty 50 (buy & hold)")
    ax1.set_title("Portfolio Equity Curve vs Nifty 50", fontsize=13)
    ax1.set_ylabel("Portfolio Value (₹)")
    ax1.legend(); ax1.grid(alpha=0.2)

    if not trades.empty:
        # 2. Drawdown
        ax2 = fig.add_subplot(gs[1, 0])
        eq   = np.array(equity_curve)
        peak = np.maximum.accumulate(eq)
        dd   = (eq - peak) / peak * 100
        ax2.fill_between(date_index, dd, 0, color="#E24B4A", alpha=0.4)
        ax2.set_title("Drawdown %", fontsize=11)
        ax2.set_ylabel("%"); ax2.grid(alpha=0.2)

        # 3. Monthly returns heatmap (simplified as bar)
        ax3 = fig.add_subplot(gs[1, 1])
        trades["Exit Date"] = pd.to_datetime(trades["Exit Date"])
        monthly = trades.groupby(trades["Exit Date"].dt.to_period("M"))["Profit"].sum()
        colors  = ["#1D9E75" if v >= 0 else "#E24B4A" for v in monthly.values]
        ax3.bar(range(len(monthly)), monthly.values, color=colors)
        ax3.set_title("Monthly P&L", fontsize=11)
        ax3.set_ylabel("₹"); ax3.grid(alpha=0.2, axis="y")
        ax3.set_xticks([]); ax3.axhline(0, color="black", lw=0.8)

        # 4. Exit reasons
        ax4 = fig.add_subplot(gs[2, 0])
        exit_counts = trades["Exit Reason"].value_counts()
        ax4.pie(exit_counts.values, labels=exit_counts.index,
                colors=["#534AB7","#1D9E75","#EF9F27","#E24B4A"],
                autopct="%1.0f%%", startangle=90)
        ax4.set_title("Exit Reason Breakdown", fontsize=11)

        # 5. Return distribution
        ax5 = fig.add_subplot(gs[2, 1])
        ax5.hist(trades["Return %"], bins=40,
                 color="#534AB7", alpha=0.7, edgecolor="white")
        ax5.axvline(0, color="#E24B4A", lw=1.5, linestyle="--")
        ax5.axvline(trades["Return %"].mean(), color="#1D9E75", lw=1.5,
                    linestyle="--", label=f'Mean: {trades["Return %"].mean():.1f}%')
        ax5.set_title("Trade Return Distribution", fontsize=11)
        ax5.set_xlabel("Return %"); ax5.legend(); ax5.grid(alpha=0.2)

    plt.suptitle("NSE Backtest — Optimized UTBot + Graham Intrinsic Value",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.savefig("backtest_results.png", dpi=150, bbox_inches="tight")
    print("Chart saved: backtest_results.png")
    plt.show()

# ─────────────────────────────────────────────
#  PRINT SUMMARY
# ─────────────────────────────────────────────
def print_summary(trades, equity_curve, date_index):
    print("\n" + "="*55)
    print("  BACKTEST RESULTS")
    print("="*55)

    final  = equity_curve[-1]
    years  = (date_index[-1] - date_index[0]).days / 365.25
    cagr   = calc_cagr(INITIAL_CAPITAL, final, years)
    mdd    = calc_max_drawdown(equity_curve)
    sharpe = calc_sharpe(equity_curve)

    print(f"  Period         : {date_index[0].date()} → {date_index[-1].date()}")
    print(f"  Initial Capital: ₹{INITIAL_CAPITAL:,.0f}")
    print(f"  Final Capital  : ₹{final:,.0f}")
    print(f"  Total Return   : {(final/INITIAL_CAPITAL - 1)*100:.1f}%")
    print(f"  CAGR           : {cagr*100:.1f}%")
    print(f"  Max Drawdown   : {mdd*100:.1f}%")
    print(f"  Sharpe Ratio   : {sharpe:.2f}")
    print(f"  Total Trades   : {len(trades)}")

    if not trades.empty:
        wins     = trades[trades["Profit"] > 0]
        losses   = trades[trades["Profit"] < 0]
        win_rate = len(wins) / len(trades)
        avg_win  = wins["Profit"].mean()   if len(wins)   > 0 else 0
        avg_loss = losses["Profit"].mean() if len(losses) > 0 else 0
        rr       = abs(avg_win / avg_loss) if avg_loss != 0 else float("inf")
        avg_hold = trades["Holding Days"].mean()

        print(f"  Win Rate       : {win_rate*100:.1f}%")
        print(f"  Avg Win        : ₹{avg_win:,.0f}")
        print(f"  Avg Loss       : ₹{avg_loss:,.0f}")
        print(f"  Risk/Reward    : {rr:.2f}")
        print(f"  Avg Hold (days): {avg_hold:.0f}")
        print(f"  Best Trade     : ₹{trades['Profit'].max():,.0f}")
        print(f"  Worst Trade    : ₹{trades['Profit'].min():,.0f}")

        print("\n  Exit Reason Breakdown:")
        for reason, cnt in trades["Exit Reason"].value_counts().items():
            avg_r = trades[trades["Exit Reason"]==reason]["Return %"].mean()
            print(f"    {reason:<25} {cnt:>4} trades  |  avg return: {avg_r:.1f}%")

        print("\n  Avg Profit by Market Cap:")
        cap_bins   = [0, 5_000, 50_000, 500_000, np.inf]
        cap_labels = ["Small Cap","Mid Cap","Large Cap","Mega Cap"]
        trades["Cap Group"] = pd.cut(trades["Market Cap Cr"],
                                     bins=cap_bins, labels=cap_labels)
        print(trades.groupby("Cap Group", observed=True)["Profit"].mean().to_string())

    print("="*55 + "\n")

# ─────────────────────────────────────────────
#  SAVE TRADE LOG
# ─────────────────────────────────────────────
def save_trades(trades):
    if trades.empty:
        print("No trades to save.")
        return
    trades = trades.sort_values("Exit Date")
    trades["Cumulative Profit"] = trades["Profit"].cumsum()
    trades["Equity"] = INITIAL_CAPITAL + trades["Cumulative Profit"]
    try:
        trades.to_excel("trade_log_optimized.xlsx", index=False)
        print("Trade log saved: trade_log_optimized.xlsx")
    except ImportError:
        trades.to_csv("trade_log_optimized.csv", index=False)
        print("Trade log saved: trade_log_optimized.csv")

# ─────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────
if __name__ == "__main__":
    print("\n" + "="*55)
    print("  NSE OPTIMIZED BACKTEST — starting")
    print("="*55 + "\n")

    # 1. Load tickers
    tickers = load_nse_tickers()
    print(f"Total NSE tickers found: {len(tickers)}")

    # 2. Parallel fetch
    all_data = load_all_data(tickers)

    if not all_data:
        print("ERROR: No data loaded. Check EQUITY_L.csv and internet connection.")
        exit(1)

    # 3. Backtest
    trades, equity_curve, date_index = run_backtest(all_data)

    # 4. Summary
    print_summary(trades, equity_curve, date_index)

    # 5. Benchmark
    nifty = fetch_nifty(date_index[0], date_index[-1])

    # 6. Save + plot
    save_trades(trades)
    plot_results(equity_curve, date_index, trades, nifty)